In [8]:
%%capture
%run script_nettoyage.ipynb

In [9]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# copie de travail
df = df_freq_totale.copy()

# colonnes binaires thématiques
colonnes_binaires = [
    'Art moderne et contemporain',
    "Arts de l'Islam",
    'Asie',
    'Couturier',
    'Design',
    'Egyptien',
    "Esclavage, société de plantation, histoire de La Réunion, colonisation, industrie sucrière",
    'Gallo-romain',
    'Grec',
    'Imprimé',
    'Inde',
    'Jeux',
    'Militaria',
    'Mode',
    'Mode et textile',
    'Musique',
    'Musique - chant - danse',
    "Mémoire de l'esclavage",
    'Numismatique',
    'Peinture',
    'Protohistoire',
    'Sciences',
    'Sciences de la nature',
    'Sciences fondamentales',
    'Sciences naturelles',
    'Technique et industrie',
    'Techniques',
    'Textile',
    'archéologie du bâti',
    'musée de société',
    'Amerique',
    'Archeologie',
    'Beaux_arts',
    'Arts_decoratifs',
    'Litterature',
    'Oceanie',
    'Sciences_techniques'
]

# variables utilisées
variables_numeriques = ['annee', 'freq_net']
variables_categorielles = ['NOMREG', 'Statut']

X = df[variables_numeriques + variables_categorielles + colonnes_binaires].copy()

# gestion des valeurs manquantes par la médiane
for col in variables_numeriques:
    X[col] = X[col].fillna(X[col].median())

for col in variables_categorielles:
    X[col] = X[col].fillna("Inconnu")

for col in colonnes_binaires:
    X[col] = X[col].fillna(0)

# prétraitement
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), variables_numeriques),
        ('cat', OneHotEncoder(handle_unknown='ignore'), variables_categorielles),
        ('bin', 'passthrough', colonnes_binaires)
    ]
)

# transformation
X_prep = preprocessor.fit_transform(X)

# choix du meilleur nombre de clusters avec silhouette
scores = {}
for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X_prep)
    scores[k] = silhouette_score(X_prep, labels)

print("Scores silhouette :", scores)

# meilleur k
best_k = max(scores, key=scores.get)
print("Meilleur nombre de clusters :", best_k)

# modèle final
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=20)
df['cluster'] = kmeans_final.fit_predict(X_prep)

# aperçu
print(df[['NOM DU MUSEE', 'annee', 'NOMREG', 'Statut', 'cluster']].head())

TypeError: Cannot convert ['2001' '2001' '2001' ... '2016' '2016' '2016'] to numeric